In [13]:
from pathlib import Path
import shutil

dataset_dir = Path("../../../Downloads/dataset_repos")

print(dataset_dir.resolve())
print(dataset_dir.exists())

/Users/senamukai/Downloads/dataset_repos
True


In [14]:
#AGENTS.mdとCLAUDE.mdがデータセットファイル内に存在する総数
agents_files = list(dataset_dir.rglob("AGENTS.md"))
claude_files = list(dataset_dir.rglob("CLAUDE.md"))

print("AGENTS:", len(agents_files))
print("CLAUDE:", len(claude_files))

AGENTS: 3703
CLAUDE: 3395


In [15]:
#AGENTS.mdとCLAUDE.mdのそれぞれに対して，https://の総数を数えるコード
def total_count_https(files):
    count = 0

    for file in files: #GitHubリポジトリのCode上を見ている
        with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
            count += f.read().count("https://")
    return count

print("AGENTS:", total_count_https(agents_files))
print("CLAUDE:", total_count_https(claude_files))        


AGENTS: 4323
CLAUDE: 1833


In [16]:
"""
AGENTS.mdとCLAUDE.mdのhttps://を1つ以上含むファイルに対して，
1ファイルごとに1としてカウントし，ファイルフォルダーを作成するプログラム
"""
agents_output = Path("./https_AGENTS")
claude_output = Path("./https_CLAUDE")

agents_output.mkdir(exist_ok = True)
claude_output.mkdir(exist_ok = True)

#AGNTS.md files
agents_count = 0

for file in agents_files:

    with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
        text = f.read()

    if "https://" in text:
        relative_path = file.relative_to(dataset_dir)
        save_path = agents_output / relative_path
        save_path.parent.mkdir(parents = True, exist_ok = True)
        shutil.copy2(file, save_path)
        agents_count += 1

#CLAUDE.md files
claude_count = 0

for file in claude_files:

    with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
        text = f.read()
        
    if "https://" in text:
        relative_path = file.relative_to(dataset_dir)
        save_path = claude_output / relative_path
        save_path.parent.mkdir(parents = True, exist_ok = True)
        shutil.copy2(file, save_path)
        claude_count += 1

print("https:// in AGENTS.md:", agents_count)
print("https:// in CLAUDE.md:", claude_count)

https:// in AGENTS.md: 917
https:// in CLAUDE.md: 635


In [17]:
"""
RQ2:トークン消費量について取り組むためのプログラム
dataset_repos内から，AGENTS.md or CLAUDE.mdのいずれかを含むリポジトリに対し，
10Mbyte以下のリポジトリを満たすか確認し，
どちらも満たした場合，抽出しそのリポジトリ丸ごとをwork2ディレクトリ下に配置するプログラム
"""

#元々のデータセット
dataset_dir = Path("../../../Downloads/dataset_repos")
output_dir = Path("../work2/under10Mbyte_AGENTS&&CLAUDE_repos")

#10Mbyteの計算
LIMIT_SIZE = 10 * 1024 * 1024

#出力フォルダ作成
output_dir.mkdir(parents = True, exist_ok = True)

#riposのサイズを求める関数
def get_dir_size(path):
    total = 0

    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size

    return total   

#カウンタ変数の初期化
copied_repo_count = 0
skipped_no_agent_file = 0
skipped_over_10mb = 0

#riposを1個ずつ探索
for repo_dir in dataset_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    #条件１: AGENTS.md or CLAUDE.md のいずれかを含むか確認
    has_agents = any(repo_dir.rglob("AGENTS.md"))
    has_claude = any(repo_dir.rglob("CLAUDE.md"))

    if not (has_agents or has_claude):
        skipped_no_agent_file += 1
        continue

    #条件2: リポジトリ全体のサイズが10Mbyte以下か確認
    repo_size = get_dir_size(repo_dir)

    if repo_size > LIMIT_SIZE:
        skipped_over_10mb += 1
        continue

    #条件1,2を満たす場合リポジトリをコピーする
    save_path = output_dir / repo_dir.name

    if save_path.exists():
        shutil.rmtree(save_path)

    shutil.copytree(
        repo_dir,
        save_path,
        symlinks = True,
        ignore_dangling_symlinks = True
    )
       
    copied_repo_count += 1

print("copied repositories:", copied_repo_count)
print("skipped no AGENTS.md or CLAUDE.md:", skipped_no_agent_file)
print("skipped over 10Mbyte:", skipped_over_10mb)


copied repositories: 3513
skipped no AGENTS.md or CLAUDE.md: 1225
skipped over 10Mbyte: 0


In [18]:
"""
dataset_repo内のリポジトリ構成がどうなっているか確認
dataset_repo内のリポジトリの中で最も容量が大きいのは何MBかを確認
コピペ(from Chat GPT)
"""

dataset_dir = Path("../../../Downloads/dataset_repos")

for i, p in enumerate(dataset_dir.iterdir()):
    print(p)
    if i == 10:
        break

#20件のリポジトリの容量を確認
for i, repo_dir in enumerate(dataset_dir.iterdir()):

    if not repo_dir.is_dir():
        continue

    size = get_dir_size(repo_dir)

    print(
        repo_dir.name,
        round(size / 1024 / 1024, 2),
        "MB"
    )

    if i == 20:
        break

#最大容量のリポジトリを求める
max_size = 0
max_repo = ""

for repo_dir in dataset_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    size = get_dir_size(repo_dir)

    if size > max_size:
        max_size = size
        max_repo = repo_dir.name

print("Largest repository")
print(max_repo)
print(round(max_size / 1024 / 1024, 2), "MB")

../../../Downloads/dataset_repos/ansys§pymapdl
../../../Downloads/dataset_repos/marcobambini§gravity
../../../Downloads/dataset_repos/cloudflare§partykit
../../../Downloads/dataset_repos/nugine§s3s
../../../Downloads/dataset_repos/dotnet§maintenance-packages
../../../Downloads/dataset_repos/giselles-ai§giselle
../../../Downloads/dataset_repos/onflow§flow-go
../../../Downloads/dataset_repos/openforis§collect-earth
../../../Downloads/dataset_repos/candid82§joker
../../../Downloads/dataset_repos/ever-co§ever-teams
../../../Downloads/dataset_repos/TrueLayer§truelayer-dotnet
ansys§pymapdl 0.02 MB
marcobambini§gravity 0.0 MB
cloudflare§partykit 0.01 MB
nugine§s3s 0.0 MB
dotnet§maintenance-packages 0.0 MB
giselles-ai§giselle 0.09 MB
onflow§flow-go 0.01 MB
openforis§collect-earth 0.01 MB
candid82§joker 0.01 MB
ever-co§ever-teams 0.5 MB
TrueLayer§truelayer-dotnet 0.0 MB
tmux-python§libtmux 0.02 MB
linuxdeepin§dde-control-center 0.0 MB
greenshot§greenshot 0.02 MB
dotnet§docker-tools 0.01 MB
adaf

In [ ]:
"""
RQ2:トークン消費量について取り組むためのプログラムの続き
上記フィルターをかけたものの内CLOUDE.mdを含むもののみを取り出して，
https://のリンクがあるかないかで分けるプログラム
"""

input_dir = Path("../work2/under10Mbyte_AGENTS&&CLAUDE_repos")
output_dir = Path("../work3")

https_output = output_dir / "https_CLAUDE"
no_https_output = output_dir / "no_https_CLAUDE"

https_output.mkdir(parents = True, exist_ok = True)
no_https_output.mkdir(parents = True, exist_ok = True)

#カウンタ変数の初期化
https_count = 0
no_https_count = 0
skipped_no_claude = 0

for repo_dir in input_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    claude_files = list(repo_dir.rglob("CLAUDE.md"))

    #CLAUDE.mdが存在しないディレクトリは対象外
    if len(claude_files) == 0:
        skipped_no_claude += 1
        continue

    #CLAUDE.mdのファイルはhttps://のリンクを含むか
    has_https = False

    for claude_file in claude_files:
        with open(claude_file, "r", encoding = "utf-8", errors = "ignore") as f:
            text = f.read()

        if "https://" in text:
            has_https = True
            break

    if has_https:
        save_path = https_output / repo_dir.name
        https_count += 1
    else:
        save_path = no_https_output / repo_dir.name
        no_https_count += 1

    if save_path.exists():
        shutil.rmtree(save_path)

    shutil.copytree(
        repo_dir,
        save_path,
        symlinks = True,
        ignore_dangling_symlinks = True
    )

print("CLAUDErepos with https:// : ", https_count)
print("CLAUDErepos without https:// : ", no_https_count)
print("skipped no CLAUDE.md : ", skipped_no_claude)